In [0]:
# ==========================================
# SILVER COMPETITION - SOURCE INSPECTION
# ==========================================

# Read the original Bronze Delta table
bronze_df = spark.table(
    "workspace.default.bronze_competition"
)

# Inspect the JSON structure
bronze_df.printSchema()

# Display the available columns
print("Columns:", bronze_df.columns)

# Preview the competition metadata
display(bronze_df)


In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp, size

# ==========================================
# 1. READ BRONZE COMPETITION
# ==========================================

bronze_df = spark.table(
    "workspace.default.bronze_competition"
)

# ==========================================
# 2. FLATTEN COMPETITION INFORMATION
# ==========================================

silver_df = bronze_df.select(

    # Competition details
    col("id").alias("competition_id"),
    col("name").alias("competition_name"),
    col("code").alias("competition_code"),
    col("type").alias("competition_type"),

    # Country information
    col("area.id").alias("country_id"),
    col("area.name").alias("country"),
    col("area.code").alias("country_code"),

    # Current season
    col("currentSeason.id").alias("season_id"),
    to_date(col("currentSeason.startDate")).alias("season_start"),
    to_date(col("currentSeason.endDate")).alias("season_end"),
    col("currentSeason.currentMatchday").alias("current_matchday"),

    # Additional metadata
    size(col("seasons")).alias("available_seasons"),
    to_timestamp(col("lastUpdated")).alias("last_updated")
)

print("Silver competition DataFrame created!")
display(silver_df)


In [0]:
# ==========================================
# 3. CLEAN AND VALIDATE
# ==========================================

# Remove records missing essential identifiers
silver_df = silver_df.dropna(
    subset=["competition_id", "season_id"]
)

# Remove duplicate competition-season records
silver_df = silver_df.dropDuplicates(
    ["competition_id", "season_id"]
)

# Count records
total_records = silver_df.count()

# Check that the table is not empty
assert total_records > 0, "No competition records found!"

# Check for invalid season dates
invalid_dates = silver_df.filter(
    col("season_start") > col("season_end")
).count()

assert invalid_dates == 0, "Invalid season dates detected!"

# Check that competition names are present
missing_names = silver_df.filter(
    col("competition_name").isNull()
).count()

assert missing_names == 0, "Missing competition names!"

print("Competition validation passed!")
print("Total clean competition records:", total_records)

# ==========================================
# 4. SAVE SILVER COMPETITION
# ==========================================

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_competition")
)

print("Silver competition table created successfully!")



In [0]:
%sql
SELECT
    competition_name,
    competition_code,
    country,
    season_id,
    season_start,
    season_end,
    current_matchday,
    available_seasons
FROM workspace.default.silver_competition;
